<a href="https://colab.research.google.com/github/Airplane356/video-background-remover/blob/main/SAM3_YOLO_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Video Background Remover using YOLO + Meta SAM3
</h1>

In [ ]:
#  ──── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

!pip install git+https://github.com/facebookresearch/sam3.git -q

!pip install ultralytics -q

!pip install torch torchvision -q

!pip install "imageio>=2.33,!=2.35.0" "huggingface-hub>=1.5.0,<2.0" -q

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print('\n✅ All dependencies installed.')


In [ ]:
# ──── USER CONFIG ──────────────────────────────────────────────────────────────────────
import os

INPUT_DIR  = '/content/videos'
OUTPUT_DIR = '/content/sam3_out'

OUTPUT_MODE   = 'greenscreen' # greenscreen, alpha, transparent

MAX_SIZE      = 0

# detection thresholds
YOLO_CONF     = 0.4
CUT_THRESHOLD = 0.20  # sensitivity for detecting scene cut (lower = more cuts)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')


In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── SAM 3 video predictor ─────────────────────────────────────────────────────
from sam3.model_builder import build_sam3_video_predictor

# build_sam3_video_predictor() downloads the checkpoint from HF Hub on first run.
sam3_video_predictor = build_sam3_video_predictor()
print('✅ SAM 3 video predictor loaded')

# ── YOLOv8n person detector ───────────────────────────────────────────────────
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')
print('✅ YOLOv8n loaded')


In [ ]:
import imageio
from tqdm import tqdm
import gc, glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────────────────────
#  1. SCENE-CUT DETECTION
# ─────────────────────────────────────────────────────────────────────────────
def detect_scene_cuts(video_path: str, threshold: float = 0.35) -> list[int]:
    """Returns frame indices where a scene cut was detected. Frame 0 always included."""
    cap = cv2.VideoCapture(video_path)
    cut_frames = [0]
    prev_hist = None
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = [cv2.calcHist([hsv], [c], None, [64], [0, 256]) for c in range(3)]
        for h in hist:
            cv2.normalize(h, h)
        if prev_hist is not None:
            diffs = [
                cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA)
                for c in range(3)
            ]
            if float(np.mean(diffs)) > threshold:
                cut_frames.append(frame_idx)
        prev_hist = hist
        frame_idx += 1
    cap.release()
    return cut_frames

# ─────────────────────────────────────────────────────────────────────────────
#  2. YOLO PERSON DETECTION
#
#  Returns a dict with:
#    'selected'  — list of [x1,y1,x2,y2] boxes that pass the score filter
#                  (these are the ones fed to SAM 3, ONE at a time)
#    'rejected'  — list of [x1,y1,x2,y2] boxes that were detected but filtered out
#    'scores'    — score for every detected box (selected + rejected order)
#
#  SAM 3 video predictor only accepts ONE bounding box per add_prompt call.
#  When multiple people pass the filter, each gets its own prompt call with
#  its own obj_id so they are tracked as separate objects but all unioned
#  into a single foreground mask.
# ─────────────────────────────────────────────────────────────────────────────
def detect_person_boxes(frame_bgr: np.ndarray, yolo_conf: float = 0.4) -> dict:
    """
    Run YOLO on frame_bgr and return a dict:
      {
        'selected': [[x1,y1,x2,y2], ...],  # boxes passed to SAM 3
        'rejected': [[x1,y1,x2,y2], ...],  # detected but filtered out
        'all_scores': [float, ...],         # score per detected box (same order as all detections)
      }
    """
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    yolo_results = yolo_model(frame_rgb, conf=yolo_conf, classes=[0], verbose=False)[0]

    if yolo_results.boxes is None or len(yolo_results.boxes) == 0:
        return {'selected': [], 'rejected': [], 'all_scores': []}

    raw_boxes = yolo_results.boxes.xyxy.cpu().numpy()
    confs     = yolo_results.boxes.conf.cpu().numpy()

    scored = []
    cx_frame, cy_frame = w / 2, h / 2
    for box, conf in zip(raw_boxes, confs):
        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1
        cx_box = (x1 + x2) / 2
        cy_box = (y1 + y2) / 2
        centrality = 1.0 - (
            abs(cx_box - cx_frame) / cx_frame * 0.5 +
            abs(cy_box - cy_frame) / cy_frame * 0.5
        )
        area_frac = (bw * bh) / (w * h)
        score = float(conf) * (1 + centrality) * (1 + area_frac * 3)
        scored.append((score, box))

    best_score = max(s for s, _ in scored)
    threshold  = 0.6 * best_score

    selected, rejected, all_scores = [], [], []
    for score, box in scored:
        all_scores.append(score)
        if score >= threshold:
            selected.append(box)
        else:
            rejected.append(box)

    return {'selected': selected, 'rejected': rejected, 'all_scores': all_scores}

# ─────────────────────────────────────────────────────────────────────────────
#  3. YOLO BOX VISUALISER  ← run this independently to inspect detection
#
#  Usage:
#    visualise_yolo_boxes(frame_bgr)           # uses YOLO_CONF from config
#    visualise_yolo_boxes(frame_bgr, conf=0.3) # override threshold
#    visualise_yolo_boxes(all_frames[0])       # first frame of loaded video
# ─────────────────────────────────────────────────────────────────────────────
def visualise_yolo_boxes(frame_bgr: np.ndarray, conf: float = None) -> None:
    """
    Display all YOLO detections on frame_bgr.

    Green boxes  = selected (will be passed to SAM 3, one per add_prompt call)
    Red boxes    = rejected (detected but scored below the 60% threshold)

    Each box is labelled with its composite score.
    Also prints a summary to stdout.
    """
    if conf is None:
        conf = YOLO_CONF

    result = detect_person_boxes(frame_bgr, yolo_conf=conf)
    selected = result['selected']
    rejected = result['rejected']

    print(f'YOLO detections (conf≥{conf}):')
    print(f'  ✓ Selected (sent to SAM 3): {len(selected)}')
    print(f'  ✗ Rejected (filtered out):  {len(rejected)}')

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(frame_rgb)

    score_idx = 0
    for box in selected:
        x1, y1, x2, y2 = box
        score = result['all_scores'][score_idx]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            boxstyle='square,pad=0',
            linewidth=2.5, edgecolor='#00ff88', facecolor='none',
        )
        ax.add_patch(rect)
        ax.text(x1 + 4, y1 - 6, f'✓ score={score:.2f}',
                color='#00ff88', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
        score_idx += 1

    for box in rejected:
        x1, y1, x2, y2 = box
        score = result['all_scores'][score_idx]
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            boxstyle='square,pad=0',
            linewidth=2, edgecolor='#ff4444', facecolor='none', linestyle='--',
        )
        ax.add_patch(rect)
        ax.text(x1 + 4, y1 - 6, f'✗ score={score:.2f}',
                color='#ff4444', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
        score_idx += 1

    green_patch = mpatches.Patch(color='#00ff88', label=f'Selected → sent to SAM 3 ({len(selected)})')
    red_patch   = mpatches.Patch(color='#ff4444', label=f'Rejected → filtered out ({len(rejected)})')
    ax.legend(handles=[green_patch, red_patch], loc='upper right', fontsize=10,
              framealpha=0.8)
    ax.set_title(f'YOLO detections  |  conf≥{conf}  |  {len(selected)+len(rejected)} person(s) found',
                 fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────
#  4. SAM 3 VIDEO PIPELINE (per shot)
#
#  SAM 3 video predictor accepts only ONE box per add_prompt call.
#  When multiple people are selected, we issue one add_prompt call per box,
#  each with a unique obj_id, then union all masks per frame.
# ─────────────────────────────────────────────────────────────────────────────
def run_sam3_on_shot(frames_bgr: list, boxes_xyxy: list, tmp_dir: str) -> list:
    """
    Returns list of np.ndarray [H, W] uint8 (0 or 255), one per frame.
    boxes_xyxy: list of [x1,y1,x2,y2] pixel-space boxes for the first frame.
    Each box is added as a separate SAM 3 prompt (one box per call, as required).
    """
    import os, shutil
    from PIL import Image as PILImage

    h, w = frames_bgr[0].shape[:2]

    # Write frames as JPEGs — SAM 3 video predictor requires a frame directory
    os.makedirs(tmp_dir, exist_ok=True)
    for fi, frame in enumerate(frames_bgr):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        PILImage.fromarray(rgb).save(os.path.join(tmp_dir, f'{fi:05d}.jpg'))

    # Convert pixel-space xyxy → normalised cxcywh before entering autocast
    norm_boxes = []
    for box in boxes_xyxy:
        x1, y1, x2, y2 = [float(v) for v in box]
        norm_boxes.append([
            (x1 + x2) / 2 / w,
            (y1 + y2) / 2 / h,
            (x2 - x1) / w,
            (y2 - y1) / h,
        ])

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):

        resp = sam3_video_predictor.handle_request(
            request=dict(type='start_session', resource_path=tmp_dir)
        )
        session_id = resp['session_id']

        # SAM 3 only accepts ONE box per add_prompt call — loop over boxes,
        # assigning each a unique obj_id so they are tracked independently.
        for obj_id, norm_box in enumerate(norm_boxes, start=1):
            sam3_video_predictor.handle_request(
                request=dict(
                    type='add_prompt',
                    session_id=session_id,
                    frame_index=0,
                    text='person',
                    bounding_boxes=[norm_box],      # exactly ONE box
                    bounding_box_labels=[1],
                )
            )

        # Propagate and collect masks
        masks_per_frame = [np.zeros((h, w), dtype=np.uint8) for _ in frames_bgr]

        for frame_resp in sam3_video_predictor.handle_stream_request(
            request=dict(type='propagate_in_video', session_id=session_id)
        ):
            fi = frame_resp['frame_index']
            outputs = frame_resp.get('outputs')
            if not outputs:
                continue
            out_masks = outputs.get('out_binary_masks')
            if out_masks is None or len(out_masks) == 0:
                continue
            combined = np.zeros((h, w), dtype=np.uint8)
            for mask in out_masks:
                m = mask.squeeze().cpu().numpy().astype(np.uint8) * 255
                if m.shape != (h, w):
                    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
                combined = cv2.bitwise_or(combined, m)
            masks_per_frame[fi] = combined

        sam3_video_predictor.handle_request(
            request=dict(type='close_session', session_id=session_id)
        )

    shutil.rmtree(tmp_dir, ignore_errors=True)
    return masks_per_frame


print('\u2705 Pipeline functions defined')
print()
print('Tip: run  visualise_yolo_boxes(all_frames[0])  to inspect YOLO boxes on any frame.')


In [ ]:
# ──── YOLO BOX INSPECTOR — run this cell independently ────────────────────────

import cv2, glob, os

# Pick a video to inspect
_videos = sorted(glob.glob(os.path.join(INPUT_DIR, '*.mp4')) +
                 glob.glob(os.path.join(INPUT_DIR, '*.mov')))
if _videos:
    _cap = cv2.VideoCapture(_videos[0])
    _frame_idx = 0          # ← change to inspect a different frame
    _cap.set(cv2.CAP_PROP_POS_FRAMES, _frame_idx)
    _, _frame = _cap.read()
    _cap.release()

    print(f'Inspecting: {os.path.basename(_videos[0])}  frame {_frame_idx}')
    visualise_yolo_boxes(_frame)          # uses YOLO_CONF from config cell
    # visualise_yolo_boxes(_frame, conf=0.3)  # ← lower conf to see more boxes
else:
    print('No videos found in INPUT_DIR — set _frame to any BGR numpy array.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE — YOLO → SAM 3 video predictor
# ─────────────────────────────────────────────────────────────────────────────

import gc, glob, os
import tempfile

GREEN = np.array([120, 255, 152], dtype=np.float32) / 255.

VIDEO_EXTENSIONS = ('*.mp4', '*.mov', '*.avi', '*.mkv')
all_videos = []
for ext in VIDEO_EXTENSIONS:
    all_videos.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
all_videos = sorted(all_videos)
print(f'Found {len(all_videos)} video(s) to process')

for video_idx, INPUT_VIDEO in enumerate(all_videos):
    base_name   = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
    output_path = os.path.join(OUTPUT_DIR, f'{base_name}_matte.mp4')

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f'\n[{video_idx+1}/{len(all_videos)}] Skipping (already done): {base_name}')
        continue

    print(f'\n{"="*70}')
    print(f'[{video_idx+1}/{len(all_videos)}] Processing: {base_name}')
    print(f'{"="*70}')

    try:
        # ── Read video metadata ───────────────────────────────────────────────
        cap = cv2.VideoCapture(INPUT_VIDEO)
        fps    = cap.get(cv2.CAP_PROP_FPS)
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f'Video: {total} frames @ {fps:.2f}fps  |  {width}×{height}')

        # ── Step 1: Scene-cut detection ───────────────────────────────────────
        print('\n[1/4] Detecting scene cuts ...')
        cut_starts = detect_scene_cuts(INPUT_VIDEO, threshold=CUT_THRESHOLD)
        shot_intervals = []
        for i, start in enumerate(cut_starts):
            end = cut_starts[i + 1] if i + 1 < len(cut_starts) else total
            shot_intervals.append((start, end))
        print(f'Found {len(shot_intervals)} shot(s): {shot_intervals}')

        # ── Step 2: Load all frames ───────────────────────────────────────────
        print('\n[2/4] Reading frames ...')
        cap = cv2.VideoCapture(INPUT_VIDEO)
        all_frames = []
        for _ in tqdm(range(total), desc='Loading frames'):
            ret, frame = cap.read()
            if not ret:
                break
            all_frames.append(frame)
        cap.release()
        print(f'Loaded {len(all_frames)} frames')

        # ── Step 3: Per-shot SAM 3 tracking ──────────────────────────────────
        print('\n[3/4] Processing shots ...')
        all_masks = []  # list of [H, W] uint8 masks, one per frame

        for shot_idx, (shot_start, shot_end) in enumerate(shot_intervals):
            shot_frames = all_frames[shot_start:shot_end]
            n_frames    = len(shot_frames)
            print(f'\n  Shot {shot_idx + 1}/{len(shot_intervals)}: '
                  f'frames {shot_start}–{shot_end - 1} ({n_frames} frames)')

            if n_frames == 0:
                continue

            # Detect person boxes on the first frame of this shot
            print(f'  Detecting persons on first frame ...')
            boxes = detect_person_boxes(shot_frames[0], yolo_conf=YOLO_CONF)

            if not boxes:
                print(f'  [WARN] No person detected in shot {shot_idx + 1} — outputting blank masks.')
                h, w = shot_frames[0].shape[:2]
                all_masks.extend([np.zeros((h, w), dtype=np.uint8)] * n_frames)
                continue

            print(f'  Found {len(boxes)} person box(es). Running SAM 3 on {n_frames} frames ...')
            tmp_dir = os.path.join('/tmp', f'sam3_shot_{shot_idx}')
            shot_masks = run_sam3_on_shot(shot_frames, boxes, tmp_dir)
            all_masks.extend(shot_masks)

            torch.cuda.empty_cache()
            gc.collect()

        print(f'\nProcessed {len(all_masks)} mask frames total')

        # ── Step 4: Composite and write output ────────────────────────────────
        print('\n[4/4] Writing output ...')
        composite_frames = []

        for frame, mask in tqdm(zip(all_frames, all_masks), total=len(all_frames), desc='Compositing'):
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
            pha = (mask / 255.)[..., None]  # [H, W, 1]

            if OUTPUT_MODE == 'greenscreen':
                comp = frame_rgb * pha + GREEN * (1.0 - pha)
            elif OUTPUT_MODE == 'alpha':
                comp = np.repeat(pha, 3, axis=2)
            else:  # transparent — write RGBA PNGs separately
                comp = frame_rgb * pha

            composite_frames.append(np.round(np.clip(comp * 255, 0, 255)).astype(np.uint8))

        tmp_output = output_path + '.tmp.mp4'
        imageio.mimwrite(tmp_output, composite_frames, fps=fps, quality=8)
        os.rename(tmp_output, output_path)

        if OUTPUT_MODE == 'transparent':
            rgba_dir = os.path.join(OUTPUT_DIR, f'{base_name}_rgba_frames')
            os.makedirs(rgba_dir, exist_ok=True)
            for fi, (frame, mask) in enumerate(zip(all_frames, all_masks)):
                rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgba = np.dstack([rgb, mask])
                Image.fromarray(rgba).save(os.path.join(rgba_dir, f'{fi:05d}.png'))
            print(f'RGBA frames saved to {rgba_dir}')

        print(f'\n✅ Done: {base_name}')
        print(f'   Output: {output_path}')

        del all_frames, all_masks, composite_frames
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        import traceback
        print(f'\n❌ FAILED: {base_name} — {e}')
        traceback.print_exc()
        print('   Continuing to next video ...')
        for tmp in [output_path + '.tmp.mp4']:
            if os.path.exists(tmp):
                os.remove(tmp)
        torch.cuda.empty_cache()
        gc.collect()
        continue

print(f'\n{"="*70}')
print('All videos processed.')


In [ ]:
import matplotlib.pyplot as plt
import math

n_shots  = len(shot_intervals)
n_cols   = min(4, n_shots)
n_rows   = math.ceil(n_shots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, (start, end) in enumerate(shot_intervals):
    if start < len(all_frames) and start < len(all_masks):
        frame_rgb = cv2.cvtColor(all_frames[start], cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
        pha = (all_masks[start] / 255.)[..., None]
        comp = frame_rgb * pha + np.array([[[0, 0.69, 0.25]]]) * (1.0 - pha)
        axes[i].imshow(np.clip(comp, 0, 1))
    axes[i].set_title(f'Shot {i+1}  (frames {start}–{end-1})', fontsize=9)
    axes[i].axis('off')

for j in range(n_shots, len(axes)):
    axes[j].axis('off')

plt.suptitle('First frame of each detected shot (green-screen composite)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shot_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Shot preview saved.')


In [ ]:
from google.colab import files

print(f'Downloading: {output_path}')
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(INPUT_VIDEO)
scores = []
prev_hist = None

while True:
    ret, frame = cap.read()
    if not ret:
        break
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hist = [cv2.calcHist([hsv], [c], None, [64], [0, 256]) for c in range(3)]
    for h in hist:
        cv2.normalize(h, h)
    if prev_hist is not None:
        diffs = [cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA) for c in range(3)]
        scores.append(float(np.mean(diffs)))
    prev_hist = hist
cap.release()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(scores, linewidth=0.8, color='steelblue', label='HSV histogram diff')
ax.axhline(CUT_THRESHOLD, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Current threshold ({CUT_THRESHOLD})')
ax.set_xlabel('Frame')
ax.set_ylabel('Bhattacharyya distance')
ax.set_title('Scene cut signal — peaks above the red line are detected as cuts')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Detected cut frames at threshold {CUT_THRESHOLD}: {detect_scene_cuts(INPUT_VIDEO, CUT_THRESHOLD)}')